In [1]:
using LinearAlgebra
import Pkg

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

DEV = true
REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
    include(joinpath(REPO_ROOT, "src", "EDKit.jl"))
    using .EDKit
else
    using EDKit
end
import EDKit: σ


## XXZ model
We consider the Hamiltonian
$$
H = \sum_i \left[
    J(\sigma_i^x \sigma_{i+1}^x + \sigma_i^y \sigma_{i+1}^y) + 
    \Delta \sigma_i^z \sigma_{i+1}^z 
\right].
$$
---
Check parity symmetry:

In [42]:
L = 10
N = 5

Js = randn(L-1)
Δs = randn(L-1)
mats = [Js[i]*(σ(1,1)+σ(2,2)) + Δs[i]*σ(3,3) for i in 1:L-1]
inds = [i:i+1 for i in 1:L-1]
B = ProjectedBasis(;L, N)
H = operator(mats, inds, B)
E, V = eigen(Hermitian(H))
S = [ent_S(V[:, i], 1:L÷2, B) for i in axes(V, 2)]

bp1 = FlipBasis(;L, p=1, N)
bp2 = FlipBasis(;L, p=-1, N)
E1, V1 = operator(mats, inds, bp1) |> Hermitian |> eigen
E2, V2 = operator(mats, inds, bp2) |> Hermitian |> eigen
S1 = [ent_S(V1[:, i], 1:L÷2, bp1) for i in axes(V1, 2)]
S2 = [ent_S(V2[:, i], 1:L÷2, bp2) for i in axes(V2, 2)]

ve = sort([E1; E2]) .- E |> norm 
vs = sort([S1; S2]) .- sort(S) |> norm 
println("----------------------------------------
Spin flip:\nE error = $ve \nS error = $vs")


Js = Js + reverse(Js)
Δs = Δs + reverse(Δs)
mats = [Js[i]*(σ(1,1)+σ(2,2)) + Δs[i]*σ(3,3) for i in 1:L-1]
inds = [i:i+1 for i in 1:L-1]
B = ProjectedBasis(;L, N)
H = operator(mats, inds, B)
E, V = eigen(Hermitian(H))
S = [ent_S(V[:, i], 1:L÷2, B) for i in axes(V, 2)]

bp1 = ParityBasis(;L, p=1, N)
bp2 = ParityBasis(;L, p=-1, N)
E1, V1 = operator(mats, inds, bp1) |> Hermitian |> eigen
E2, V2 = operator(mats, inds, bp2) |> Hermitian |> eigen
S1 = [ent_S(V1[:, i], 1:L÷2, bp1) for i in axes(V1, 2)]
S2 = [ent_S(V2[:, i], 1:L÷2, bp2) for i in axes(V2, 2)]

ve = sort([E1; E2]) .- E |> norm
vs = sort([S1; S2]) .- sort(S) |> norm

println("----------------------------------------
Parity:\nE error = $ve \nS error = $vs")

bp1 = ParityFlipBasis(;L, p=1, z=1, N)
bp2 = ParityFlipBasis(;L, p=1, z=-1, N)
bp3 = ParityFlipBasis(;L, p=-1, z=1, N)
bp4 = ParityFlipBasis(;L, p=-1, z=-1, N)
E1, V1 = operator(mats, inds, bp1) |> Hermitian |> eigen
E2, V2 = operator(mats, inds, bp2) |> Hermitian |> eigen
E3, V3 = operator(mats, inds, bp3) |> Hermitian |> eigen
E4, V4 = operator(mats, inds, bp4) |> Hermitian |> eigen
S1 = [ent_S(V1[:, i], 1:L÷2, bp1) for i in axes(V1, 2)]
S2 = [ent_S(V2[:, i], 1:L÷2, bp2) for i in axes(V2, 2)]
S3 = [ent_S(V3[:, i], 1:L÷2, bp3) for i in axes(V3, 2)]
S4 = [ent_S(V4[:, i], 1:L÷2, bp4) for i in axes(V4, 2)]

ve = sort([E1; E2; E3; E4]) .- E |> norm
vs = sort([S1; S2; S3; S4]) .- sort(S) |> norm
println("----------------------------------------
Spin flip + Parity:\nE error = $ve \nS error = $vs")

----------------------------------------
Spin flip:
E error = 6.624190125204214e-14 
S error = 1.1511167688078701e-12
----------------------------------------
Parity:
E error = 1.2792024185418167e-13 
S error = 5.985644681356852e-13
----------------------------------------
Spin flip + Parity:
E error = 1.5983911384436993e-13 
S error = 1.0317365109538752e-12
